# LAB | First match ≠ right match
## Relevance Scoring & Rerankers for Trustworthy AI + EU AI Act

**Learning goal:** build one retrieval path over two sources and prove every answer is grounded in evidence you can inspect.

**Sources in the workspace root**

- `podcast.txt`: a podcast transcript about the non-binding 2019 Ethics Guidelines for Trustworthy AI.
- `eu_ai_act.pdf`: Regulation (EU) 2024/1689, the binding EU AI Act.

---
## Step 0: Environment [SOLUTION]

---
## Environment dependencies [DOCUMENTATION ONLY]

The runtime dependencies used by this notebook are listed below. This cell is intentionally markdown so **Run All never installs or upgrades packages**. Install them manually in the selected environment only when needed.

```text
python-dotenv pymupdf cohere langchain-core langchain-community
langchain-text-splitters langchain-openai pinecone langchain-pinecone
```

In [1]:
import os
import re
import time
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

for key in ["OPENAI_API_KEY", "PINECONE_API_KEY"]:
    if not os.getenv(key):
        raise ValueError(f"{key} not found. Add it to .env before a live run.")

print("OpenAI and Pinecone credentials are configured.")
print(f"Cohere reranking configured: {bool(os.getenv('COHERE_API_KEY'))}")

OpenAI and Pinecone credentials are configured.
Cohere reranking configured: True


---
## Step 1: Setup & Data Preparation [SOLUTION]

The two source types use the same chunking rule but retain traceable metadata. PDF pages are converted from the loader's 0-based index to the human-facing 1-based page number. The conservative `section` detector only labels an explicit heading found inside a chunk; ambiguous PDF chunks receive `other`, never `None`.

### 1a. Load both sources

In [2]:
from langchain_community.document_loaders import PyMuPDFLoader, TextLoader

PODCAST_PATH = Path("podcast.txt")
PDF_PATH = Path("eu_ai_act.pdf")

for path in [PODCAST_PATH, PDF_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Missing workspace source: {path}")

podcast_docs = TextLoader(str(PODCAST_PATH), encoding="utf-8").load()
pdf_docs = PyMuPDFLoader(str(PDF_PATH)).load()

print(f"Podcast: {len(podcast_docs)} document, {len(podcast_docs[0].page_content):,} characters")
print(f"EU AI Act: {len(pdf_docs)} pages")
print("Raw metadata from the PDF loader:", pdf_docs[0].metadata)

C:\Users\maxke\AppData\Local\Temp\ipykernel_13632\1936117646.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader, TextLoader


Podcast: 1 document, 16,414 characters
EU AI Act: 144 pages
Raw metadata from the PDF loader: {'producer': 'PDFlib+PDI 9.0.7p3 (C++/Win64)', 'creator': 'Servigistics Arbortext Publishing Engine', 'creationdate': '2024-07-11T14:47:17+02:00', 'source': 'eu_ai_act.pdf', 'file_path': 'eu_ai_act.pdf', 'total_pages': 144, 'format': 'PDF 1.7', 'title': 'Regulation (EU) 2024/1689 of the European Parliament and of the Council of 13 June 2024 laying down harmonised rules on artificial intelligence and amending Regulations (EC) No 300/2008, (EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and Directives 2014/90/EU, (EU) 2016/797 and (EU) 2020/1828 (Artificial Intelligence Act)Text with EEA relevance.', 'author': 'Publications Office of the European Union L-2985 Luxembourg LUXEMBOURG', 'subject': 'I Legislative acts', 'keywords': 'ISSN 1977-0677', 'moddate': '2024-07-11T15:55:28+02:00', 'trapped': '', 'modDate': "D:20240711155528+02'00'", 'creationDate': "D:2024

### 1b. Chunk with deterministic metadata

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

def detect_pdf_section(text):
    """Return a section only when an explicit heading is present in this chunk."""
    heading_lines = [line.strip() for line in text.splitlines() if line.strip()]
    for line in heading_lines:
        if re.match(r"^(?:ANNEX|Annex)\s+[IVX\d]+\b", line):
            return "annex"
        if re.match(r"^(?:Article|ARTICLE)\s+\d+\b", line):
            return "article"
        if re.match(r"^(?:Recital|RECITALS?)\b", line):
            return "recital"
    return "other"

def chunk_with_metadata(docs, source_type, source_name):
    """Split documents and attach only stable, non-null Pinecone metadata."""
    chunks = splitter.split_documents(docs)
    for i, chunk in enumerate(chunks):
        metadata = {
            "source_type": source_type,
            "source_name": source_name,
            "chunk_id": f"{source_type}-{i}",
        }
        if source_type == "regulation":
            raw_page = chunk.metadata.get("page")
            if raw_page is not None:
                metadata["page"] = int(raw_page) + 1
            metadata["section"] = detect_pdf_section(chunk.page_content)
        chunk.metadata = metadata
    return chunks

podcast_chunks = chunk_with_metadata(podcast_docs, "podcast", "Trustworthy AI podcast")
act_chunks = chunk_with_metadata(pdf_docs, "regulation", "EU AI Act (Reg. 2024/1689)")
all_chunks = podcast_chunks + act_chunks

print(f"Podcast chunks:   {len(podcast_chunks)}")
print(f"EU AI Act chunks: {len(act_chunks)}")
print(f"Total chunks:     {len(all_chunks)}")

Podcast chunks:   20
EU AI Act chunks: 763
Total chunks:     783


### 1c. Inspect representative chunks

In [ ]:
for chunk in [podcast_chunks[0], act_chunks[min(150, len(act_chunks) - 1)]]:
    print(chunk.metadata)
    print(chunk.page_content[:300], "...")
    print("-" * 80)

### 1d. Embeddings model

In [5]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print("Embedding model initialized.")

Embedding model initialized.


---
## Step 2: Pinecone baseline retrieval [SOLUTION]

The lab uses the existing-or-create `lab-rerankers` index with a dedicated namespace. `RUN_INDEXING` defaults to `False`; `RUN_DEMOS` also defaults to `False` so running all cells does not issue the five baseline queries or filter demonstrations. A populated namespace is usable read-only when indexing is disabled, while indexing is refused when it would mix or overwrite an existing namespace.

> **Reindexing note:** changing the PDF extractor can change chunk text and IDs' content. Use a fresh dedicated namespace and set `RUN_INDEXING = True` for a fresh lab-namespace upsert before comparing retrieval results, then return it to `False`.


In [8]:
# Fixed queries: reuse these unchanged for baseline, reranked, and evaluation views.
TEST_QUERIES = [
    "What AI practices are prohibited under the EU AI Act?",
    "What are the seven requirements for trustworthy AI?",
    "What is the difference between human-in-the-loop and human-on-the-loop?",
    "What fines apply for using a prohibited AI practice?",
    "Is social scoring of citizens allowed?",
]

def show_results(query, results, label="Results", score_label="pinecone_similarity"):
    """Display source, location, and a short verbatim excerpt."""
    print(f"\n=== {label} | Query: {query}")
    for rank, (doc, score) in enumerate(results, 1):
        metadata = doc.metadata
        location = f"p.{metadata['page']}" if "page" in metadata else metadata["chunk_id"]
        print(
            f"#{rank}  {score_label}={score:.3f}  "
            f"[{metadata['source_type']} | {location} | {metadata['chunk_id']}]  "
            f"{doc.page_content[:180]!r}"
        )

In [7]:
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

INDEX_NAME = "lab-rerankers"
INDEX_DIMENSION = 1536
INDEX_NAMESPACE = "relevance-lab-pymupdf-v1"
RUN_INDEXING = False
RUN_DEMOS = False
RUN_EVALUATION = False

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
existing_indexes = set(pc.list_indexes().names())
vectorstore = None
INDEX_READY = False

def _field_value(value, name, default=None):
    if hasattr(value, name):
        return getattr(value, name)
    if isinstance(value, dict):
        return value.get(name, default)
    return default

def _wait_for_index_ready(pinecone_client, index_name, timeout_seconds=120):
    """Wait for a newly created index before describing stats or using it."""
    started = time.monotonic()
    while True:
        description = pinecone_client.describe_index(index_name)
        status = _field_value(description, "status", {}) or {}
        if _field_value(status, "ready") is True:
            return description
        if time.monotonic() - started >= timeout_seconds:
            raise TimeoutError(f"Pinecone index {index_name!r} was not ready within {timeout_seconds}s.")
        time.sleep(1)

def _namespace_vector_count(index, namespace):
    stats = index.describe_index_stats()
    namespaces = _field_value(stats, "namespaces", {}) or {}
    entry = namespaces.get(namespace, {}) if isinstance(namespaces, dict) else {}
    return int(_field_value(entry, "vector_count", 0) or 0)

if INDEX_NAME not in existing_indexes:
    if RUN_INDEXING:
        pc.create_index(
            name=INDEX_NAME,
            dimension=INDEX_DIMENSION,
            metric="cosine",
            spec=ServerlessSpec(cloud="aws", region="us-east-1"),
        )
        index_description = _wait_for_index_ready(pc, INDEX_NAME)
        print(f"Created and confirmed ready: {INDEX_NAME}")
    else:
        print(
            f"Index {INDEX_NAME!r} does not exist and RUN_INDEXING=False; "
            "no index was created. Set RUN_INDEXING=True for the first live setup."
        )
else:
    index_description = pc.describe_index(INDEX_NAME)

if INDEX_NAME in existing_indexes or RUN_INDEXING:
    index_dimension = _field_value(index_description, "dimension")
    if index_dimension is None:
        raise ValueError(f"Could not read the dimension of index {INDEX_NAME!r}.")
    if int(index_dimension) != INDEX_DIMENSION:
        raise ValueError(
            f"Index {INDEX_NAME!r} has dimension {index_dimension}; expected {INDEX_DIMENSION}. "
            "Use a compatible lab index rather than changing a shared index."
        )

    pinecone_index = pc.Index(INDEX_NAME)
    namespace_count = _namespace_vector_count(pinecone_index, INDEX_NAMESPACE)
    if namespace_count and RUN_INDEXING:
        print(
            f"Namespace {INDEX_NAMESPACE!r} contains {namespace_count} vectors; "
            "reindexing is blocked. Set RUN_INDEXING=False for read-only retrieval "
            "or choose a fresh namespace. No data was deleted."
        )
    elif namespace_count:
        vectorstore = PineconeVectorStore(
            index_name=INDEX_NAME,
            embedding=embeddings,
            namespace=INDEX_NAMESPACE,
        )
        INDEX_READY = True
        print(f"Using populated namespace {INDEX_NAMESPACE!r} read-only; no uploads performed.")
    elif RUN_INDEXING:
        vectorstore = PineconeVectorStore(
            index_name=INDEX_NAME,
            embedding=embeddings,
            namespace=INDEX_NAMESPACE,
        )
        stable_ids = [chunk.metadata["chunk_id"] for chunk in all_chunks]
        vectorstore.add_documents(all_chunks, ids=stable_ids)
        INDEX_READY = True
        print(f"Upserted {len(all_chunks)} stable chunks into namespace {INDEX_NAMESPACE!r}.")
    else:
        print(
            f"Namespace {INDEX_NAMESPACE!r} is empty and RUN_INDEXING=False; "
            "no uploads performed. Set RUN_INDEXING=True for initial indexing."
        )


Using populated namespace 'relevance-lab-pymupdf-v1' read-only; no uploads performed.


In [9]:
def baseline_search(query, k=5, metadata_filter=None):
    """Return Pinecone (Document, similarity) pairs with an optional server-side filter."""
    if not INDEX_READY or vectorstore is None:
        raise RuntimeError(
            "Baseline retrieval is unavailable: index/namespace is not ready for this lab run."
        )
    if not query or not query.strip():
        return []
    return vectorstore.similarity_search_with_score(
        query,
        k=k,
        filter=metadata_filter,
    )

# This loop is intentionally disabled by default; set RUN_DEMOS=True for an explicit live run.
baseline_results = {}
if RUN_DEMOS and INDEX_READY:
    for query in TEST_QUERIES:
        baseline_results[query] = baseline_search(query, k=5)
        show_results(query, baseline_results[query], label="Baseline")
elif not RUN_DEMOS:
    print("Baseline demo disabled; set RUN_DEMOS=True to run the five fixed queries.")
else:
    print("Baseline demo skipped because the index/namespace is not ready.")


Baseline demo disabled; set RUN_DEMOS=True to run the five fixed queries.


In [10]:
query = None
baseline_candidates = []

if RUN_DEMOS and INDEX_READY:
    query = TEST_QUERIES[0]
    baseline_candidates = baseline_search(query, k=10)
    show_results(
        query,
        baseline_candidates[:3],
        label="Baseline Top 3",
    )
elif not RUN_DEMOS:
    print("Baseline inspection disabled; set RUN_DEMOS=True to run it explicitly.")
else:
    print("Baseline inspection skipped because the index/namespace is not ready.")


=== Baseline Top 3 | Query: What AI practices are prohibited under the EU AI Act?
#1  pinecone_similarity=0.697  [regulation | p.51.0 | regulation-341]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a sufficient level of AI literacy of \ntheir staff and other persons'
#2  pinecone_similarity=0.681  [regulation | p.9.0 | regulation-57]  'intention to distort behaviour where the distortion results from factors external to the AI system which are outside \nthe control of the provider or the deployer, namely factors th'
#3  pinecone_similarity=0.680  [regulation | p.12.0 | regulation-83]  '(45)\nPractices that are prohibited by Union law, including data protection law, non-discrimination law, consumer \nprotection law, and competition law, should not be affected by thi'


---
## Step 3: Optional per-chunk LLM relevance scoring [SKIPPED]

This path is intentionally omitted: it would add one LLM call per candidate and is not needed to meet the lab objective. The required comparison uses one dedicated Cohere reranker, and Cohere scores are kept separate from Pinecone similarity scores.

---
## Step 4: One dedicated reranker — Cohere

Cohere receives the exact initial candidate texts in one request. It returns the original `Document` objects in its relevance order; no score averaging or score blending is performed.

In [11]:
COHERE_RERANK_MODEL = os.getenv("COHERE_RERANK_MODEL", "rerank-v5.0-fast")

def _read_field(value, name, default=None):
    if hasattr(value, name):
        return getattr(value, name)
    if isinstance(value, dict):
        return value.get(name, default)
    return default

def _rerank_candidates(query, candidates, k_final=5):
    if not candidates:
        return []
    api_key = os.getenv("COHERE_API_KEY")
    if not api_key:
        raise RuntimeError("COHERE_API_KEY is required to run the Cohere reranker.")
    try:
        import cohere
    except ImportError as exc:
        raise RuntimeError("Install the cohere package before running reranking.") from exc

    client = cohere.ClientV2(api_key=api_key)
    documents = [doc.page_content for doc, _ in candidates]
    response = client.rerank(
        model=COHERE_RERANK_MODEL,
        query=query,
        documents=documents,
        top_n=min(k_final, len(documents)),
    )

    reranked = []
    for item in (_read_field(response, "results", []) or []):
        index = _read_field(item, "index")
        relevance_score = _read_field(item, "relevance_score")
        if not isinstance(index, int) or not 0 <= index < len(candidates):
            continue
        try:
            relevance_score = float(relevance_score)
        except (TypeError, ValueError):
            continue
        reranked.append((candidates[index][0], relevance_score))

    if not reranked:
        raise RuntimeError("Cohere returned no valid reranking results.")
    return reranked[:k_final]

def cohere_rerank(query, k_initial=10, k_final=5):
    """Retrieve once, send all candidate texts in one request, and return Cohere scores."""
    candidates = baseline_search(query, k=k_initial)
    return _rerank_candidates(query, candidates, k_final=k_final)

In [12]:
def show_top3_comparison(query, candidates=None, k_initial=10, k_final=5):
    """Compare top 3 using one shared initial candidate list."""
    shared_candidates = candidates if candidates is not None else baseline_search(query, k=k_initial)
    baseline_top3 = shared_candidates[:3]
    reranked_top3 = _rerank_candidates(query, shared_candidates, k_final=min(k_final, 3))
    show_results(query, baseline_top3, label="Baseline top 3 from shared initial candidates (Pinecone similarity)")
    show_results(
        query,
        reranked_top3,
        label="Cohere reranked top 3 (Cohere relevance)",
        score_label="cohere_relevance",
    )
    return {"query": query, "candidates": shared_candidates, "baseline_top3": baseline_top3, "reranked_top3": reranked_top3}

# Set execute=True only during a credentialed live run; this remains unexecuted for offline validation.
def compare_fixed_queries(execute=False):
    if not execute:
        print("Comparison prepared for TEST_QUERIES[:2]; call compare_fixed_queries(execute=True) during a live run.")
        return []
    return [show_top3_comparison(query, candidates=baseline_search(query, k=10)) for query in TEST_QUERIES[:2]]

compare_fixed_queries(execute=False)

Comparison prepared for TEST_QUERIES[:2]; call compare_fixed_queries(execute=True) during a live run.


[]

In [13]:
if RUN_DEMOS and INDEX_READY:
    comparison = show_top3_comparison(
        query,
        candidates=baseline_candidates,
    )
elif not RUN_DEMOS:
    print("Reranking inspection disabled; set RUN_DEMOS=True to run it explicitly.")
else:
    print("Reranking inspection skipped because the index/namespace is not ready.")


=== Baseline top 3 from shared initial candidates (Pinecone similarity) | Query: What AI practices are prohibited under the EU AI Act?
#1  pinecone_similarity=0.697  [regulation | p.51.0 | regulation-341]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a sufficient level of AI literacy of \ntheir staff and other persons'
#2  pinecone_similarity=0.681  [regulation | p.9.0 | regulation-57]  'intention to distort behaviour where the distortion results from factors external to the AI system which are outside \nthe control of the provider or the deployer, namely factors th'
#3  pinecone_similarity=0.680  [regulation | p.12.0 | regulation-83]  '(45)\nPractices that are prohibited by Union law, including data protection law, non-discrimination law, consumer \nprotection law, and competition law, should not be affected by thi'

=== Cohere reranked top 3 (Cohere relevance) | Query: What AI practices are prohibited under the E

In [14]:
if RUN_DEMOS and INDEX_READY:
    for rank, (doc, score) in enumerate(baseline_candidates, 1):
        preview = " ".join(doc.page_content.split())[:180]
        print(
            f"#{rank} | page={doc.metadata.get('page')} "
            f"| {doc.metadata.get('chunk_id')} "
            f"| {preview}"
        )
elif not RUN_DEMOS:
    print("Candidate inspection disabled; set RUN_DEMOS=True to run it explicitly.")
else:
    print("Candidate inspection skipped because the index/namespace is not ready.")

#1 | page=51.0 | regulation-341 | Article 4 AI literacy Providers and deployers of AI systems shall take measures to ensure, to their best extent, a sufficient level of AI literacy of their staff and other persons 
#2 | page=9.0 | regulation-57 | intention to distort behaviour where the distortion results from factors external to the AI system which are outside the control of the provider or the deployer, namely factors tha
#3 | page=12.0 | regulation-83 | (45) Practices that are prohibited by Union law, including data protection law, non-discrimination law, consumer protection law, and competition law, should not be affected by this
#4 | page=45.0 | regulation-312 | (b) prohibitions of certain AI practices; (c) specific requirements for high-risk AI systems and obligations for operators of such systems; (d) harmonised transparency rules for ce
#5 | page=3.0 | regulation-14 | operators involved in the AI value chain under this Regulation should apply without prejudice to national law, 

In [15]:
if RUN_DEMOS and INDEX_READY and baseline_candidates:
    print(baseline_candidates[0][0].page_content)
elif not RUN_DEMOS:
    print("Chunk inspection disabled; set RUN_DEMOS=True to run it explicitly.")
else:
    print("Chunk inspection skipped because no demo candidates are available.")

Article 4
AI literacy
Providers and deployers of AI systems shall take measures to ensure, to their best extent, a sufficient level of AI literacy of 
their staff and other persons dealing with the operation and use of AI systems on their behalf, taking into account their 
technical knowledge, experience, education and training and the context the AI systems are to be used in, and considering 
the persons or groups of persons on whom the AI systems are to be used.
CHAPTER II
PROHIBITED AI PRACTICES
Article 5
Prohibited AI practices
1.
The following AI practices shall be prohibited:
(a) the placing on the market, the putting into service or the use of an AI system that deploys subliminal techniques beyond 
a person’s consciousness or purposefully manipulative or deceptive techniques, with the objective, or the effect of 
materially distorting the behaviour of a person or a group of persons by appreciably impairing their ability to make an


In [16]:
if RUN_DEMOS and INDEX_READY:
    article_results = baseline_search(
        query,
        k=3,
        metadata_filter={"section": {"$eq": "article"}},
    )
    show_results(
        query,
        article_results,
        label="Article Metadata Filter",
    )
elif not RUN_DEMOS:
    print("Article metadata inspection disabled; set RUN_DEMOS=True to run it explicitly.")
else:
    print("Article metadata inspection skipped because the index/namespace is not ready.")


=== Article Metadata Filter | Query: What AI practices are prohibited under the EU AI Act?
#1  pinecone_similarity=0.697  [regulation | p.51.0 | regulation-341]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a sufficient level of AI literacy of \ntheir staff and other persons'
#2  pinecone_similarity=0.670  [regulation | p.45.0 | regulation-312]  '(b) prohibitions of certain AI practices;\n(c) specific requirements for high-risk AI systems and obligations for operators of such systems;\n(d) harmonised transparency rules for ce'
#3  pinecone_similarity=0.626  [regulation | p.92.0 | regulation-551]  '3.\nParagraph 1 is without prejudice to Union or national law which excludes processing of personal data for other \npurposes than those explicitly mentioned in that law, as well as '


---
## Step 5: Pinecone metadata filtering

Filters are sent to Pinecone through `similarity_search_with_score(..., filter=...)`, so these are database-side filters rather than client-side list filtering. The explicit-heading detector makes `section="article"` safe to query for chunks that contain an Article heading; ambiguous chunks remain `other`.

In [ ]:
if RUN_DEMOS and INDEX_READY:
    FILTER_QUERY = TEST_QUERIES[0]
    regulation_results = baseline_search(
        FILTER_QUERY,
        k=5,
        metadata_filter={"source_type": {"$eq": "regulation"}},
    )
    podcast_results = baseline_search(
        FILTER_QUERY,
        k=5,
        metadata_filter={"source_type": {"$eq": "podcast"}},
    )
    show_results(FILTER_QUERY, regulation_results, label="Pinecone filter: source_type=regulation")
    show_results(FILTER_QUERY, podcast_results, label="Pinecone filter: source_type=podcast")
elif not RUN_DEMOS:
    print("Source-type filter demo disabled; set RUN_DEMOS=True to run it explicitly.")
else:
    print("Source-type filter demo skipped because the index/namespace is not ready.")


In [ ]:
if RUN_DEMOS and INDEX_READY:
    article_results = baseline_search(
        FILTER_QUERY,
        k=5,
        metadata_filter={"section": {"$eq": "article"}},
    )
    show_results(FILTER_QUERY, article_results, label="Pinecone filter: section=article")
elif not RUN_DEMOS:
    print("Section filter demo disabled; set RUN_DEMOS=True to run it explicitly.")
else:
    print("Section filter demo skipped because the index/namespace is not ready.")


---
## Step 6: Grounded RAG with a swappable retriever

`answer(query, retrieve_fn=...)` returns the generated text plus the exact `(Document, score)` pairs and context sent to the model. Switch from baseline to Cohere by passing `retrieve_fn=cohere_rerank`; the prompt and answer path stay unchanged.

In [17]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

def _citation_label(metadata):
    location = f"p.{metadata['page']}" if "page" in metadata else metadata["chunk_id"]
    return f"[{metadata['source_name']}, {location}, {metadata['chunk_id']}]"

def format_context(results):
    """Build inspectable context while preserving each source excerpt verbatim."""
    return "\n\n".join(
        f"{_citation_label(doc.metadata)}\n{doc.page_content}"
        for doc, _score in results
    )

rag_llm = ChatOpenAI(model=os.getenv("CHAT_MODEL", "gpt-4o-mini"), temperature=0)
rag_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """Answer the question only from the Evidence below. Cite every substantive claim with the exact bracketed source identifier shown in Evidence. If the evidence is insufficient, say exactly: I don't know based on the provided sources.

The EU AI Act is binding legislation. The podcast discusses the non-binding 2019 Ethics Guidelines for Trustworthy AI. Keep those sources distinct and do not invent legal claims, article numbers, penalties, or requirements.

Evidence:
{context}""",
    ),
    ("human", "Question: {question}"),
])

def answer(query, retrieve_fn=baseline_search):
    results = retrieve_fn(query)
    context = format_context(results)
    chain = rag_prompt | rag_llm | StrOutputParser()
    generated_answer = chain.invoke({"question": query, "context": context})
    return {
        "query": query,
        "answer": generated_answer,
        "results": results,
        "context": context,
    }

---
## Step 7: Evaluation Results [EXECUTED]

The table below records the saved five-question evaluation outputs. `Top-1 relevant?` refers to the first retrieved chunk for that method. `Grounded` is marked `Y` only when the answer's cited chunk appears in the retrieved evidence; model abstentions use `N/A` as requested.

| Query | Method | Top-1 relevant? (Y/N) | Answer correct? (Y/Partial/N/N/A) | Grounded in cited chunk? (Y/N/N/A) | Notes |
|---|---|---:|---:|---:|---|
| Q1 prohibited AI practices | Baseline | N | Partial | N | Top-1 was `regulation-341` (Article 4 AI literacy), not the prohibition text; the answer included relevant claims but was not grounded in that top result. |
| Q1 prohibited AI practices | Cohere reranked | N | Partial | N | Cohere kept the same unrelated top-1 chunk; the revised answer was not consistently better grounded. |
| Q2 seven trustworthy-AI requirements | Baseline | N | N/A | N/A | Abstained. Top-1 `podcast-2` introduced the topic but did not provide the seven-item list. |
| Q2 seven trustworthy-AI requirements | Cohere reranked | N | N/A | N/A | Abstained. Top-1 `regulation-49` mentioned the 2019 guidelines but still did not retrieve the requirements list. |
| Q3 human-in/on-the-loop | Baseline | Y | Y | Y | `podcast-8` directly supports the distinction; the answer cited the retrieved podcast evidence. |
| Q3 human-in/on-the-loop | Cohere reranked | Y | Y | Y | Same directly relevant top-1 and an accurate grounded answer. |
| Q4 fines for prohibited practice | Baseline | N | Y | Y | Top-1 `regulation-341` was unrelated, but rank 3 `regulation-664` supported the cited EUR 35M/7% answer. |
| Q4 fines for prohibited practice | Cohere reranked | Y | Y | Y | Cohere moved `regulation-664` to top-1 and preserved the correct grounded answer. |
| Q5 social scoring | Baseline | Y | Y | Y | Top-1 `regulation-60` was directly about social-scoring data; the answer cited retrieved `regulation-343`. |
| Q5 social scoring | Cohere reranked | N | Y | Y | Top-1 shifted to related biometric-inference text, but the answer cited retrieved `regulation-60`. |

**Conclusion:** Cohere did not consistently improve final answer quality. It clearly improved the fines result's top-1 retrieval, left the human-in/on-the-loop result effectively unchanged, and did not resolve the prohibited-practices grounding issue or the seven-requirements failure.

**Observed failure or limitation:** the system could not answer the question about the seven Trustworthy AI requirements because neither baseline retrieval nor Cohere reranking surfaced a chunk containing the complete seven-item list; both correctly abstained rather than inventing it.

In [18]:
def evaluate_fixed_queries(execute=False):
    """Prepare manual comparison data; no network or model calls unless execute=True."""
    if not execute:
        print("Evaluation helper prepared for all five TEST_QUERIES; call evaluate_fixed_queries(execute=True) during a live run.")
        return []

    evaluations = []
    for query in TEST_QUERIES:
        started = time.perf_counter()
        shared_candidates = baseline_search(query, k=10)
        baseline_retriever = lambda _query, results=shared_candidates[:5]: results
        baseline_started = time.perf_counter()
        baseline_output = answer(query, retrieve_fn=baseline_retriever)
        baseline_elapsed = time.perf_counter() - baseline_started

        rerank_started = time.perf_counter()
        reranked_results = _rerank_candidates(query, shared_candidates, k_final=5)
        rerank_elapsed = time.perf_counter() - rerank_started
        reranked_retriever = lambda _query, results=reranked_results: results
        reranked_answer_started = time.perf_counter()
        reranked_output = answer(query, retrieve_fn=reranked_retriever)
        reranked_answer_elapsed = time.perf_counter() - reranked_answer_started

        show_results(query, shared_candidates[:5], label="Baseline evaluation results (Pinecone similarity)")
        show_results(
            query,
            reranked_results,
            label="Cohere evaluation results (Cohere relevance)",
            score_label="cohere_relevance",
        )
        print("Baseline answer:", baseline_output["answer"])
        print("Reranked answer:", reranked_output["answer"])

        evaluations.append({
            "query": query,
            "shared_initial_candidates": shared_candidates,
            "baseline": baseline_output,
            "reranked": reranked_output,
            "timing_seconds": {
                "total": time.perf_counter() - started,
                "baseline_answer": baseline_elapsed,
                "cohere_rerank": rerank_elapsed,
                "reranked_answer": reranked_answer_elapsed,
            },
        })
    return evaluations

evaluate_fixed_queries(execute=False)

Evaluation helper prepared for all five TEST_QUERIES; call evaluate_fixed_queries(execute=True) during a live run.


[]

In [19]:
evaluation_results = None
if RUN_EVALUATION:
    evaluation_results = evaluate_fixed_queries(execute=True)
else:
    print("Evaluation disabled; set RUN_EVALUATION=True to execute the five-question evaluation explicitly.")


=== Baseline evaluation results (Pinecone similarity) | Query: What AI practices are prohibited under the EU AI Act?
#1  pinecone_similarity=0.697  [regulation | p.51.0 | regulation-341]  'Article 4\nAI literacy\nProviders and deployers of AI systems shall take measures to ensure, to their best extent, a sufficient level of AI literacy of \ntheir staff and other persons'
#2  pinecone_similarity=0.681  [regulation | p.9.0 | regulation-57]  'intention to distort behaviour where the distortion results from factors external to the AI system which are outside \nthe control of the provider or the deployer, namely factors th'
#3  pinecone_similarity=0.680  [regulation | p.12.0 | regulation-83]  '(45)\nPractices that are prohibited by Union law, including data protection law, non-discrimination law, consumer \nprotection law, and competition law, should not be affected by thi'
#4  pinecone_similarity=0.670  [regulation | p.45.0 | regulation-312]  '(b) prohibitions of certain AI practices;\n(c

---
## Submission boundary

Do not write `lab_proof.md` from this notebook. After real runtime tests, the student should add it with one query, retrieved evidence, the final cited answer, a claim-to-chunk trace, baseline-versus-Cohere comparison, and one observed failure or limitation. Runtime indexing, remote APIs, answer generation, and retrieval-quality judgments are intentionally not verified by offline validation.